# SME Capital Matching — Funding Readiness Model & Segmentation

Taking the cleaned data file produced by the `Data_Cleaning` notebook, this notebook scores every SME on funding readiness and segments the results into three tiers. Each business receives a **Funding Readiness Score from 0 to 100** and is placed into a tier — **High**, **Mid** or **Low** readiness — so the funding team knows who to progress now, who to develop, and who to revisit later.

The *Steps* are as follows:
1. [Find the `Datasets` folder, load the clean data, and set the pillar weights;](#step-1)
2. [Score Pillar 1 — Revenue & Growth;](#step-2)
3. [Score Pillar 2 — Jobs & Scale;](#step-3)
4. [Score Pillar 3 — Funding-Ask Viability;](#step-4)
5. [Score Pillar 4 — Transformation / B-BBEE;](#step-5)
6. [Combine the four pillars into the 0–100 score;](#step-6)
7. [Assign a tier to every business and rank them;](#step-7)
8. [Review the top of the list;](#step-8)
9. [Build a styled Excel workbook and save it to the `Datasets` folder.](#step-9)

**How the score is built.** Every business is measured against four pillars. Each pillar produces a number between 0 and 1, that number is multiplied by the pillar's weight, and the four results are added together to give the score out of 100:

| Pillar | Weight | The question it answers |
|---|---|---|
| 1. Revenue & Growth | **30 pts** | Is there a real, growing business here? |
| 2. Jobs & Scale | **25 pts** | Is there operating traction and job-creation impact? |
| 3. Funding-Ask Viability | **25 pts** | Is the amount asked for sensible, and is the form complete enough to assess? |
| 4. Transformation (B-BBEE) | **20 pts** | BEE level and inclusive ownership |

Each pillar step below sets out the exact formula and then works through a real business from the dataset, so the calculation can be followed end to end and checked against the final spreadsheet.

**A fairness rule applied throughout:** a *missing* answer is treated as **neutral** (a middle score of 0.5), never as zero. No business is penalised for a blank the form failed to capture. The two deliberate exceptions, where a blank genuinely carries meaning, are called out in Steps 2 and 3.

| Phase | File | Folder |
|---|---|---|
| The tidy, analysis-ready table | `Capital_Matching_Cleaned_Data.xlsx` | `Datasets` |
| ⬇ *this notebook inputs, scores and tiers it, then outputs* | `Funding_Readiness_Segmentation.ipynb` | `Python_Notebooks` |
| **Final result** — scored, tiered and ranked | `Funding_Readiness_Segmentation.xlsx` | `Datasets` |

> **Data note.** This notebook is committed with a sanitised dataset that mirrors the shape of the live application data. The tier counts produced below (**273 / 568 / 275**) are therefore illustrative, and will not match the figures quoted in the README and the insights deck (**245 / 595 / 276**), which come from the confidential live dataset. The scoring model, pillar weights and tier thresholds are identical in both cases — only the underlying records differ.

<a id="step-1"></a>
## Step 1 — Find the `Datasets` folder, load the clean data, and set the pillar weights

This step performs three actions.

**1. Locate the `Datasets` folder.** Search the current folder, then each folder above it, until the folder named `Datasets` is found — the same method used in the `Data_Cleaning` notebook.

**2. Load the clean data.** Open `Capital_Matching_Cleaned_Data.xlsx`, the output of the `Data_Cleaning` notebook, from that folder. If the file is not present, the notebook stops with a clear instruction to run the `Data_Cleaning` notebook first, rather than failing with a confusing error.

**3. Set the pillar weights.** Declare the four pillar weights up front, in one place, so that the model can be re-tuned by changing four numbers and nothing else.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# 1. Locate the Datasets folder.
def find_datasets_folder():
    """Look in this folder, then the one above it, and so on, until 'Datasets' is found."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "Datasets").is_dir():
            return folder / "Datasets"
    raise FileNotFoundError(
        "Could not find a folder named 'Datasets'. Please open this notebook from "
        "inside the project folder (the one that contains 'Datasets')."
    )

# Set the file paths: the clean input file, and the segmentation output file saved in Step 9.
DATASETS = find_datasets_folder()
CLEAN_FILE = DATASETS / "Capital_Matching_Cleaned_Data.xlsx"
SEGMENTATION_FILE = DATASETS / "Funding_Readiness_Segmentation.xlsx"

# Stop with a clear message if the Data_Cleaning notebook has not been run yet.
if not CLEAN_FILE.exists():
    raise FileNotFoundError(
        f"'{CLEAN_FILE.name}' is not in the Datasets folder yet.\n"
        "Please run the Data_Cleaning notebook first - it creates this file."
    )

print(f"Datasets folder : {DATASETS}")
print(f"Reading from    : {CLEAN_FILE.name}")
print(f"Will save to    : {SEGMENTATION_FILE.name}")

# 2. Load the clean data.
df = pd.read_excel(CLEAN_FILE)
print(f"\nLoaded {len(df):,} clean businesses")

# 3. Set the four pillar weights (they must total 100).
WEIGHTS = {
    "revenue_growth": 30,   # Pillar 1
    "jobs_scale":     25,   # Pillar 2
    "ask_viability":  25,   # Pillar 3
    "transformation": 20,   # Pillar 4
}
assert sum(WEIGHTS.values()) == 100, "weights must total 100"

# Helper used by every pillar: keep a score safely inside the 0-to-1 range.
def clip01(s):
    """Keep a score safely inside the 0..1 range."""
    return s.clip(lower=0, upper=1)

Datasets folder : C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets
Reading from    : Capital_Matching_Cleaned_Data.xlsx
Will save to    : Funding_Readiness_Segmentation.xlsx



Loaded 1,116 clean businesses


<a id="step-2"></a>
## Step 2 — Score Pillar 1: Revenue & Growth (30 points)

This pillar measures two things: the size of the business's most recent annual revenue, and the direction of that revenue between 2023 and 2024. Each component is calculated as a number between 0 and 1.

**Size — 60% of the pillar.** The `Data_Cleaning` notebook converted each annual revenue band into a position on a scale from 0 (`0 to R500K`) up to 6 (`Above R50M`). The size score divides that position by 6:

> **size = latest band position ÷ 6**

The 2024 band is used; if 2024 is blank, the 2023 band is used instead. A business with no band in either year is treated as pre-revenue and scores 0. This is the first of the two deliberate exceptions to the missing-is-neutral rule: a business that reports no revenue at all is read as not yet earning, rather than as unknown.

**Growth — 40% of the pillar.** Band movement is the number of positions the business moved between its 2023 band and its 2024 band, capped at two bands in either direction so that a single unusual year cannot dominate. The movement is converted to a 0-to-1 scale where no movement is exactly the middle:

> **growth = (band movement + 2) ÷ 4**, with movement capped between −2 and +2

Moving up one band gives (1 + 2) ÷ 4 = 0.75, no movement gives 0.5, and moving down one band gives 0.25. If either year's band is blank, movement cannot be calculated, so the business receives the neutral 0.5.

**The combination:** pillar = 0.60 × size + 0.40 × growth, multiplied by the 30-point weight.

**Example from the data — Vuka Transport CC** (ranked 1st of 1,116). Its 2023 band was `R20 000 001 to R50M` (position 5) and its 2024 band `Above R50M` (position 6):

- size = 6 ÷ 6 = **1.0**
- growth: up one band, so (1 + 2) ÷ 4 = **0.75**
- pillar = 0.60 × 1.0 + 0.40 × 0.75 = 0.90 → 0.90 × 30 = **27.0 points**

That 27.0 is exactly the value in the `pts_revenue_growth` column of the final spreadsheet, which is how any of these calculations can be audited.

In [2]:
# Pillar 1: score revenue size and revenue growth.
def score_revenue_growth(df):
    # 1. Size = latest band position / 6, using 2024 and falling back to 2023.
    #    No band in either year = pre-revenue = 0.
    latest_scale = df["revenue_scale_2024"].fillna(df["revenue_scale_2023"])
    size = (latest_scale / 6.0).fillna(0.0)

    # 2. Growth = (band movement + 2) / 4, with movement capped between -2 and +2,
    #    so that +1 band = 0.75, no movement = 0.5, -1 band = 0.25.
    delta = df["revenue_scale_2024"] - df["revenue_scale_2023"]
    growth = ((delta.clip(-2, 2)) + 2) / 4.0
    growth = growth.fillna(0.5)            # movement unknown = neutral 0.5

    # Blend the two parts: pillar = 0.60 x size + 0.40 x growth.
    pillar = clip01(0.60 * size + 0.40 * growth)
    df["p1_revenue_size"] = size
    df["p1_revenue_growth"] = growth
    df["pillar_revenue_growth"] = pillar
    return pillar

score_revenue_growth(df)
print("Pillar 1 done. Average (0-1):", round(df["pillar_revenue_growth"].mean(), 3))

Pillar 1 done. Average (0-1): 0.319


<a id="step-3"></a>
## Step 3 — Score Pillar 2: Jobs & Scale (25 points)

This pillar measures two things: the number of people the business currently employs, which is direct evidence that it operates, and the number of new jobs the applicant states the funding will create. The two components are **existing employees (55%)** and **anticipated new jobs (45%)**. Both counts are converted with the same formula:

> **score = log(count + 1) ÷ log(51)**

A logarithm is used instead of a straight line for a specific reason: the difference between 1 and 5 employees is more significant, as evidence about a small business, than the difference between 50 and 54 employees, yet a straight-line scale would treat both differences as equal. The logarithmic curve increases quickly at low counts and flattens at high counts, and it is calibrated so that a count of 50 produces the full score of 1.0.

**The combination:** pillar = 0.55 × employee score + 0.45 × anticipated-jobs score, capped at 1.0, multiplied by the 25-point weight. Counts above 50 produce a component score above 1.0, which is why the cap on the combined result exists.

On this pillar, a blank answer counts as zero rather than neutral — the second deliberate exception to the missing-is-neutral rule. An unstated headcount is read as no employees recorded, and job-creation figures are not assumed on an applicant's behalf.

**Example from the data — Blue Crane Enterprises CC**, which recorded 3 employees and 3 anticipated jobs:

- each count converts to log(3 + 1) ÷ log(51) = 1.386 ÷ 3.932 = **0.35**
- pillar = 0.55 × 0.35 + 0.45 × 0.35 = 0.35 → 0.35 × 25 = **8.8 points**

By comparison, **Vuka Transport CC** recorded 144 employees and 14 anticipated jobs. Its employee score is 1.27 (144 is above the ceiling of 50) and its jobs score is log(15) ÷ log(51) = 0.69. The combined result, 0.55 × 1.27 + 0.45 × 0.69 = 1.01, is capped at 1.0 — the full **25.0 points** shown in its `pts_jobs_scale` column.

In [3]:
# Pillar 2: score operating traction and job-creation impact, using log scaling.
def score_jobs_scale(df):
    # 1. Treat blank employee and job counts as zero, and remove negative values.
    emp = df["employees"].fillna(0).clip(lower=0)
    jobs = df["jobs_anticipated"].fillna(0).clip(lower=0)

    # 2. Convert each count: score = log(count + 1) / log(51), so a count of 50 = 1.0.
    #    Counts above 50 exceed 1.0 here; the final blend below is capped back to 1.0.
    emp_norm = np.log1p(emp) / np.log1p(50)
    jobs_norm = np.log1p(jobs) / np.log1p(50)

    # Blend the two parts: pillar = 0.55 x employees + 0.45 x anticipated jobs, capped at 1.0.
    pillar = clip01(0.55 * emp_norm + 0.45 * jobs_norm)
    df["p2_employees_norm"] = clip01(emp_norm)
    df["p2_jobs_norm"] = clip01(jobs_norm)
    df["pillar_jobs_scale"] = pillar
    return pillar

score_jobs_scale(df)
print("Pillar 2 done. Average (0-1):", round(df["pillar_jobs_scale"].mean(), 3))

Pillar 2 done. Average (0-1): 0.409


<a id="step-4"></a>
## Step 4 — Score Pillar 3: Funding-Ask Viability (25 points)

This pillar measures whether a funder could act on the request as written. It has two components.

**Proportionate request — 60% of the pillar.** Each business is assigned a reference revenue: its actual 2025 revenue where one was captured, otherwise the Rand midpoint of its 2024 band, otherwise the midpoint of its 2023 band. The funding request is divided by that reference:

> **ratio = funding ask ÷ reference revenue**

The ratio is scored by three rules:

- between **0.5× and 5×** revenue: the target range, scoring the full **1.0** — a request of between half a year's and five years' revenue is the range within which funders can realistically structure a transaction;
- **below 0.5×**: score = ratio ÷ 0.5, with a minimum of 0.4 — small requests remain fundable, they are simply modest relative to the business;
- **above 5×**: score = 1 − (ratio − 5) ÷ 20 — the score decreases steadily and reaches zero at 25× revenue, because a business requesting many multiples of its revenue is presenting a plan a funder cannot verify against its trading history.

A business with no usable revenue reference keeps the neutral 0.5.

**Completeness — 40% of the pillar.** Ten fields required for assessment are checked — registration number, industry, province, request amount, funding type, funding purpose, 2024 revenue band, employees, company overview and BEE level — and the score is the number of completed fields divided by ten.

**The combination:** pillar = 0.60 × request score + 0.40 × completeness, multiplied by the 25-point weight.

**Example from the data — Vuka Transport CC** requested R40,940,000 against actual 2025 revenue of R34,115,739. The ratio is 1.2×, inside the target range, so its request scores 1.0; all ten fields are completed, so completeness is 1.0. The pillar is 1.0 → the full **25.0 points**.

**Example from the data — Blue Crane Enterprises CC** requested R90,000 against 2025 revenue of R8,639 — a ratio of 10.4×. That is above 5×, so its request scores 1 − (10.4 − 5) ÷ 20 = **0.73**. Three of its ten required fields are blank (industry, funding type and company overview), so completeness is 7 ÷ 10 = **0.70**. The pillar is 0.60 × 0.73 + 0.40 × 0.70 = 0.72 → **17.9 points**.

In [4]:
# Pillar 3: score the size of the ask against revenue, and the completeness of the application.
def score_ask_viability(df):
    # 1. Set the reference revenue: actual 2025 revenue, else the 2024 band midpoint, else the 2023 midpoint.
    ref_rev = df["revenue_2025_zar"]
    ref_rev = ref_rev.where(ref_rev > 0, df["revenue_mid_2024"])
    ref_rev = ref_rev.where(ref_rev > 0, df["revenue_mid_2023"])

    # 2. Ratio = funding ask / reference revenue.
    ask = df["funding_ask_zar"]
    ratio = ask / ref_rev

    # 3. Score the ratio: 0.5x-5x = 1.0; below 0.5x = ratio/0.5 with a floor of 0.4;
    #    above 5x the score falls steadily and reaches 0 at 25x revenue.
    def ratio_score(r):
        if pd.isna(r) or r <= 0:
            return np.nan
        if 0.5 <= r <= 5:
            return 1.0
        if r < 0.5:
            return max(0.4, r / 0.5)
        return max(0.0, 1 - (r - 5) / 20.0)

    sweet = ratio.map(ratio_score).fillna(0.5)   # no usable revenue reference = neutral 0.5

    # 4. Completeness = the share of these ten key investor fields that are filled in.
    key_fields = ["company_reg_no", "industry", "province", "funding_ask_zar", "funding_type",
                  "funding_purpose", "revenue_band_2024", "employees", "company_overview", "bee_level"]
    completeness = df[key_fields].notna().mean(axis=1)

    # Blend the two parts: pillar = 0.60 x ask score + 0.40 x completeness.
    pillar = clip01(0.60 * sweet + 0.40 * completeness)
    df["ask_to_revenue_ratio"] = ratio
    df["p3_ask_sweetspot"] = sweet
    df["p3_completeness"] = completeness
    df["pillar_ask_viability"] = pillar
    return pillar

score_ask_viability(df)
print("Pillar 3 done. Average (0-1):", round(df["pillar_ask_viability"].mean(), 3))

Pillar 3 done. Average (0-1): 0.841


<a id="step-5"></a>
## Step 5 — Score Pillar 4: Transformation / B-BBEE (20 points)

South African funders — in particular development finance institutions and corporate enterprise-development funds — are required by their mandates to consider transformation credentials, so these credentials are scored as a separate pillar. It has two equally weighted components.

**BEE level — 50% of the pillar.** Under B-BBEE, Level 1 is the strongest rating and Level 8 the weakest. The level is converted to a 0-to-1 score:

> **BEE score = (8 − level) ÷ 7**

Level 1 gives (8 − 1) ÷ 7 = 1.0, Level 4 gives approximately 0.57, and Level 8 gives 0. A missing level keeps the neutral 0.5.

**Inclusive ownership — 50% of the pillar.** The application captured four ownership percentages — Black, women, youth and disability ownership — as percentage ranges, which the `Data_Cleaning` notebook converted to midpoint values. This component is the average of the four midpoints, divided by 100:

> **ownership = average of the four ownership percentages ÷ 100**

**The combination:** pillar = 0.50 × BEE score + 0.50 × ownership, multiplied by the 20-point weight.

**Example from the data — Vuka Transport CC** holds BEE Level 1, so its BEE score is (8 − 1) ÷ 7 = **1.0**. Its ownership answers were 100% Black, 100% youth, 0% women and 0% disability, which average to 50% → **0.5**. The pillar is 0.50 × 1.0 + 0.50 × 0.5 = 0.75 → **15.0 points**.

This pillar carries the lowest weight for a measurable reason: the applicant pool is already highly transformed — 92% of applicants are majority Black-owned and 86% hold BEE Level 1 — so the pillar separates one applicant from another less than the other three. It is retained because it is a genuine requirement in many funders' mandates.

In [5]:
# Pillar 4: score BEE level and inclusive ownership.
def score_transformation(df):
    # 1. BEE score = (8 - level) / 7, so Level 1 = 1.0 and Level 8 = 0. Missing level = neutral 0.5.
    bee = df["bee_level"]
    bee_score = ((8 - bee) / 7.0).fillna(0.5).clip(0, 1)

    # 2. Ownership = average of the four ownership percentages / 100. Missing = neutral 0.5.
    own_cols = ["black_ownership_pct", "women_ownership_pct", "youth_ownership_pct", "disability_ownership_pct"]
    ownership = (df[own_cols].mean(axis=1) / 100.0).fillna(0.5).clip(0, 1)

    # Blend the two parts equally: pillar = 0.50 x BEE score + 0.50 x ownership.
    pillar = clip01(0.50 * bee_score + 0.50 * ownership)
    df["p4_bee_score"] = bee_score
    df["p4_ownership"] = ownership
    df["pillar_transformation"] = pillar
    return pillar

score_transformation(df)
print("Pillar 4 done. Average (0-1):", round(df["pillar_transformation"].mean(), 3))

Pillar 4 done. Average (0-1): 0.703


<a id="step-6"></a>
## Step 6 — Combine the four pillars into the 0–100 score

Each pillar is a number between 0 and 1. This step multiplies each one by its weight and adds the four results together:

> **score = pillar 1 × 30 + pillar 2 × 25 + pillar 3 × 25 + pillar 4 × 20**

Each pillar's point contribution is also stored in its own column (`pts_revenue_growth`, `pts_jobs_scale`, `pts_ask_viability`, `pts_transformation`), so that it is always possible to state exactly where a business gained or lost points.

**The two worked examples, brought together:**

| Pillar | Vuka Transport CC | Blue Crane Enterprises CC |
|---|---|---|
| 1. Revenue & Growth (30) | 0.90 × 30 = 27.0 | 0.20 × 30 = 6.0 |
| 2. Jobs & Scale (25) | 1.00 × 25 = 25.0 | 0.35 × 25 = 8.8 |
| 3. Funding-Ask Viability (25) | 1.00 × 25 = 25.0 | 0.72 × 25 = 17.9 |
| 4. Transformation (20) | 0.75 × 20 = 15.0 | 0.75 × 20 = 15.0 |
| **Funding Readiness Score** | **92.0** | **47.8** |

One small technical note: the score is calculated from the unrounded pillar values and rounded once at the end, while the four `pts_` columns are each rounded for display. Adding the displayed columns can therefore differ from the score by up to 0.1, as it does for Blue Crane (6.0 + 8.8 + 17.9 + 15.0 = 47.7 against a true score of 47.8).

In [6]:
# 1. Multiply each pillar by its weight and sum into the 0-100 Funding Readiness Score.
p1 = df["pillar_revenue_growth"]; p2 = df["pillar_jobs_scale"]
p3 = df["pillar_ask_viability"];  p4 = df["pillar_transformation"]

df["funding_readiness_score"] = (
    p1 * WEIGHTS["revenue_growth"] + p2 * WEIGHTS["jobs_scale"]
    + p3 * WEIGHTS["ask_viability"] + p4 * WEIGHTS["transformation"]
).round(1)

# 2. Record each pillar's point contribution separately.
df["pts_revenue_growth"] = (p1 * WEIGHTS["revenue_growth"]).round(1)
df["pts_jobs_scale"]     = (p2 * WEIGHTS["jobs_scale"]).round(1)
df["pts_ask_viability"]  = (p3 * WEIGHTS["ask_viability"]).round(1)
df["pts_transformation"] = (p4 * WEIGHTS["transformation"]).round(1)

# 3. Display the score distribution.
print("Score summary (0-100):")
print(df["funding_readiness_score"].describe().round(1).to_string())

Score summary (0-100):
count    1116.0
mean       54.9
std        11.8
min        16.5
25%        48.2
50%        54.2
75%        61.7
max        92.0


<a id="step-7"></a>
## Step 7 — Assign a tier to every business, and rank them

This step converts the score into three tiers using fixed thresholds. Fixed thresholds mean a tier keeps the same definition when new applications are scored later:

- **High readiness — score of 62 or above**: the strongest candidates, ready to be introduced to investors.
- **Mid readiness — 48 to 61**: operating businesses with specific, correctable gaps; develop and re-score.
- **Low readiness — below 48**: early-stage or incomplete applications; correct the application before any introduction.

From the examples: Vuka Transport CC, at 92.0, is above the 62 threshold and is assigned to **High**. Blue Crane Enterprises CC, at 47.8, is 0.2 points below the Mid threshold and is assigned to **Low** — its request of 10.4× revenue and its three blank required fields are exactly the correctable problems the Low tier identifies.

Every business also receives a rank, where 1 = most ready. Businesses on the same score share the same rank, and the table is sorted best-first, with ties settled alphabetically by company name so that the same data always produces exactly the same file.

In [7]:
# 1. Assign each business to a tier using the fixed score thresholds.
def tier(score):
    if score >= 62:
        return "High"
    if score >= 48:
        return "Mid"
    return "Low"

df["readiness_tier"] = pd.Categorical(
    df["funding_readiness_score"].map(tier), categories=["High", "Mid", "Low"], ordered=True
)

# 2. Rank every business, where 1 = most ready.
df["readiness_rank"] = df["funding_readiness_score"].rank(ascending=False, method="min").astype(int)

# 3. Sort with the best at the top. Tied scores are settled alphabetically by company name,
#    so that the same data always produces the same file.
df = df.sort_values(
    ["funding_readiness_score", "company_name"], ascending=[False, True]
).reset_index(drop=True)

# 4. Display the number of businesses in each tier.
tier_counts = df["readiness_tier"].value_counts().reindex(["High", "Mid", "Low"])
tier_table = pd.DataFrame({
    "Tier": tier_counts.index,
    "Businesses": tier_counts.values,
    "Share": [f"{v/len(df)*100:.1f}%" for v in tier_counts.values],
})
display(tier_table)

,Tier,Businesses,Share
0,High,273,24.5%
1,Mid,568,50.9%
2,Low,275,24.6%


<a id="step-8"></a>
## Step 8 — Review the top of the list

Before saving, this step performs a sanity check on the five most funding-ready businesses in the applicant pool. If the names at the top look like plausible, substantial businesses, the model is behaving sensibly.

In [8]:
# Display the five highest-ranked businesses.
print("Top 5 most funding-ready businesses:")
display(df[["readiness_rank", "company_name", "industry", "province",
            "funding_readiness_score", "readiness_tier"]].head(5))

Top 5 most funding-ready businesses:


,readiness_rank,company_name,industry,province,funding_readiness_score,readiness_tier
0,1,Vuka Transport CC,NPO & NPC,Gauteng,92.0,High
1,2,Maluleke Agri Group,Mining,Limpopo,90.1,High
2,3,Ubuntu Distributors CC,Education,Mpumalanga,90.0,High
3,4,Rakoma Engineering CC,Agriculture - Primary,Limpopo,89.6,High
4,5,Clear Water Foods (Pty) Ltd,Information & Communication Technology,Gauteng,89.4,High


<a id="step-9"></a>
## Step 9 — Build a styled Excel workbook and save it to the `Datasets` folder

This step saves the segmentation as a single, focused Excel table — `Funding_Readiness_Segmentation.xlsx` — back into the same `Datasets` folder the clean data was read from.

The workbook is deliberately not the full clean dataset. It keeps only the fields that matter for funding readiness: each business's identity, the inputs the model uses (revenue, jobs, ask, transformation), each pillar's point contribution, and the final score, tier and rank. One sheet, one table, colour-coded by tier. All explanation and summarised figures live in the methodology document, not here.

**This is the end of the workflow.** The `Datasets` folder now holds all three files in the chain: the raw applications, the cleaned data, and this finished segmentation — the file the dashboard and the funding team work from.

In [9]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 1. Define a helper that strips the invisible control characters Excel refuses to store.
#    Normal text and emoji are kept.
def clean_cell(v):
    if not isinstance(v, str):
        return v
    return "".join(ch for ch in v if ch in "\t\n\r" or (
        ord(ch) >= 0x20 and ord(ch) != 0x7f and not (0x80 <= ord(ch) <= 0x9f)
        and not (0xFDD0 <= ord(ch) <= 0xFDEF) and (ord(ch) & 0xFFFF) not in (0xFFFE, 0xFFFF)))

# 2. Keep only the funding-readiness-relevant columns, in a sensible reading order.
seg_cols = [
    "readiness_rank", "readiness_tier", "funding_readiness_score", "company_name",
    "industry", "province", "city_town", "funding_type", "funding_ask_zar",
    "revenue_band_2023", "revenue_band_2024", "revenue_2025_zar",
    "employees", "jobs_anticipated", "bee_level",
    "black_ownership_pct", "women_ownership_pct", "youth_ownership_pct", "disability_ownership_pct",
    "ask_to_revenue_ratio",
    "pts_revenue_growth", "pts_jobs_scale", "pts_ask_viability", "pts_transformation",
]
seg_cols = [c for c in seg_cols if c in df.columns]
seg = df[seg_cols].copy()

# 3. Set the workbook styling: font, colours and cell borders.
NAVY="1F3864"; BLUE="2E5496"; FONT="Arial"
GREEN="C6EFCE"; YELLOW="FFEB9C"; RED="FFC7CE"; GREEN_T="006100"; YELLOW_T="9C6500"; RED_T="9C0006"
thin=Side(style="thin",color="D9D9D9"); BORDER=Border(left=thin,right=thin,top=thin,bottom=thin)

# 4. Create the workbook and write the styled header row.
wb=Workbook(); ws=wb.active; ws.title="Funding Readiness"; ws.sheet_view.showGridLines=False
for j,col in enumerate(seg.columns,1):
    cell=ws.cell(row=1,column=j,value=str(col))
    cell.font=Font(name=FONT,bold=True,color="FFFFFF",size=10)
    cell.fill=PatternFill("solid",fgColor=BLUE)
    cell.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True); cell.border=BORDER

# 5. Write the data rows, applying number formats and colour-coding the tier cell green, amber or red.
for i,(_,r) in enumerate(seg.iterrows(),2):
    for j,col in enumerate(seg.columns,1):
        v=r[col]
        if pd.isna(v): v=None
        elif isinstance(v,(np.integer,)): v=int(v)
        elif isinstance(v,(np.floating,)): v=float(v)
        cell=ws.cell(row=i,column=j,value=clean_cell(v))
        cell.font=Font(name=FONT,size=9); cell.border=BORDER; cell.alignment=Alignment(vertical="center")
        if col=="funding_ask_zar" or col=="revenue_2025_zar": cell.number_format='#,##0;(#,##0);-'
        if col=="funding_readiness_score": cell.number_format='0.0'
        if col=="ask_to_revenue_ratio": cell.number_format='0.0"x"'
    tcell=ws.cell(row=i,column=2)
    if tcell.value=="High": tcell.fill=PatternFill("solid",fgColor=GREEN); tcell.font=Font(name=FONT,size=9,bold=True,color=GREEN_T)
    elif tcell.value=="Mid": tcell.fill=PatternFill("solid",fgColor=YELLOW); tcell.font=Font(name=FONT,size=9,bold=True,color=YELLOW_T)
    elif tcell.value=="Low": tcell.fill=PatternFill("solid",fgColor=RED); tcell.font=Font(name=FONT,size=9,bold=True,color=RED_T)

# 6. Freeze the header row and set sensible column widths.
ws.freeze_panes="A2"
for c in ws.columns:
    letter=get_column_letter(c[0].column)
    length=max((len(str(cell.value)) if cell.value is not None else 0) for cell in c)
    ws.column_dimensions[letter].width=min(max(length+2,10),34)

# 7. Save the workbook to the Datasets folder, using the path set in Step 1.
wb.save(SEGMENTATION_FILE)
print(f"Saved the funding readiness table -> {SEGMENTATION_FILE}")
print(f"({seg.shape[0]:,} rows x {seg.shape[1]} columns)")
print("\nWorkflow complete. The Datasets folder now holds:")
print("  1. capital_matching_applications.csv    (raw input)")
print("  2. Capital_Matching_Cleaned_Data.xlsx   (from the Data_Cleaning notebook)")
print("  3. Funding_Readiness_Segmentation.xlsx  (from this notebook)")

Saved the funding readiness table -> C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets\Funding_Readiness_Segmentation.xlsx
(1,116 rows x 24 columns)

Workflow complete. The Datasets folder now holds:
  1. capital_matching_applications.csv    (raw input)
  2. Capital_Matching_Cleaned_Data.xlsx   (from the Data_Cleaning notebook)
  3. Funding_Readiness_Segmentation.xlsx  (from this notebook)
